<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자기지도 학습 · 멀티모달 · 07. ALIGN: noisy alt-text를 규모로 이기기

## Scaling Up Visual and Vision-Language Representation Learning With Noisy Text Supervision

- **원 논문:** [Scaling Up Visual and Vision-Language Representation Learning With Noisy Text Supervision](https://arxiv.org/abs/2102.05918)
- **저자 / 발표:** Chao Jia et al. · ICML (2021)
- **난이도 / 예상 시간:** 중급 · 약 65분
- **선수 지식:** CLIP, noisy labels, normalized softmax
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 일부 caption을 의도적으로 오염시키고 ALIGN의 단순 dual-encoder 대칭 normalized-softmax를 학습합니다.

**축소하거나 생략한 부분:** 18억 noisy image-alt-text pairs 대신 240개 중 30%를 결정적으로 오염시킵니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2.1 | minimal filtering의 noisy pairs | 오염 index·비율 |
| §2.2, Eq. (1) | image-to-text normalized softmax | row CE |
| §2.2, Eq. (2) | text-to-image symmetric loss | column CE |
| Figure 1 | image/text dual encoder | L2-normalized embedding shape |
| §3.2 retrieval evaluation | dual-encoder nearest-neighbor retrieval | clean test Recall@1 |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** noisy image/text batch [B,*] → embeddings [B,D] → logits [B,B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\mathcal L=\tfrac12\left[-\log p(y_i\mid x_i)-\log p(x_i\mid y_i)\right]$$

- **기호 정의:** x_i/y_i는 noisy image-alt-text pair이며 batch의 다른 pair가 negative입니다.
- **수식의 역할:** 복잡한 fusion 없이 normalized-softmax와 데이터 규모로 noisy supervision을 견딥니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 §2.1을 구현합니다.
- **입출력 shape:** image feature [B, 64], token [B, L] → normalized embedding과 [B, B] score
- **평가:** clean test Recall@1, pair noise 비율과 shuffled-caption 비교
- **원문 대비 한계:** 18억 noisy image-alt-text pairs 대신 240개 중 30%를 결정적으로 오염시킵니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2.1
- **이 셀의 책임:** minimal filtering의 noisy pairs
- **Tensor shape 계약:** noisy image/text batch [B,*] → embeddings [B,D] → logits [B,B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 오염 index·비율. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def noisy_captions(text: Tensor, rate=0.3, seed=5):
    rng = np.random.default_rng(seed)
    out = text.clone()
    count = int(len(text) * rate)
    idx = torch.tensor(rng.choice(len(text), count, replace=False))
    out[idx] = out[idx.roll(1)]
    return out, idx


noisy, corrupted = noisy_captions(tokens[train_idx])
assert len(corrupted) == int(len(train_idx) * 0.3) and not torch.equal(
    noisy, tokens[train_idx]
)

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §2.2, Eq. (1) / §2.2, Eq. (2) / Figure 1
- **이 셀의 책임:** image-to-text normalized softmax / text-to-image symmetric loss / image/text dual encoder
- **Tensor shape 계약:** noisy image/text batch [B,*] → embeddings [B,D] → logits [B,B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** row CE / column CE / L2-normalized embedding shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class TinyALIGN(nn.Module):
    def __init__(self, vocab, dim=16):
        super().__init__()
        self.image = nn.Sequential(
            nn.Flatten(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, dim)
        )
        self.word = nn.Embedding(vocab, 20)
        self.text = nn.Linear(20, dim)
        self.temperature = nn.Parameter(torch.tensor(0.2))

    def forward(self, x, t):
        zi = F.normalize(self.image(x), dim=1)
        zt = F.normalize(self.text(self.word(t).mean(1)), dim=1)
        return zi @ zt.T / self.temperature.clamp(0.03, 1)


align = TinyALIGN(int(tokens.max()) + 1)
logits = align(images[:16], tokens[:16])
target = torch.arange(16)
loss = (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target)) / 2
assert logits.shape == (16, 16) and torch.isfinite(loss)

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §2.1 / §2.2, Eq. (1) / §2.2, Eq. (2) / Figure 1 / §3.2 retrieval evaluation
- **이 셀의 책임:** minimal filtering의 noisy pairs / image-to-text normalized softmax / text-to-image symmetric loss / image/text dual encoder / dual-encoder nearest-neighbor retrieval
- **Tensor shape 계약:** noisy image/text batch [B,*] → embeddings [B,D] → logits [B,B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 오염 index·비율 / row CE / column CE / L2-normalized embedding shape / clean test Recall@1. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
align = ACCELERATOR.move(align)
x, noisy_device = ACCELERATOR.move(images[train_idx], noisy)
optimizer = torch.optim.Adam(align.parameters(), lr=0.012)
losses = []
for _ in range(130):
    logits = align(x, noisy_device)
    target = torch.arange(len(x), device=logits.device)
    loss = (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target)) / 2
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
with torch.no_grad():
    test_images, test_tokens = ACCELERATOR.move(images[test_idx], tokens[test_idx])
    test_logits = align(test_images, test_tokens)
    recall = float(
        (
            test_logits.argmax(1)
            == torch.arange(len(test_idx), device=test_logits.device)
        )
        .float()
        .mean()
    )
assert losses[-1] < losses[0] and recall >= 1 / len(test_idx)
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(losses)
ax[0].set_title("noisy ALIGN loss")
ax[1].imshow(test_logits.detach().cpu(), cmap="viridis")
ax[1].set_title("clean test logits")
plt.tight_layout()
plt.show()
print(
    {
        "noise_rate": len(corrupted) / len(train_idx),
        "clean_retrieval@1": round(recall, 3),
    }
)

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2.1 / §2.2, Eq. (1) / §2.2, Eq. (2) / Figure 1
- **이 셀의 책임:** minimal filtering의 noisy pairs / image-to-text normalized softmax / text-to-image symmetric loss / image/text dual encoder
- **Tensor shape 계약:** noisy image/text batch [B,*] → embeddings [B,D] → logits [B,B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 오염 index·비율 / row CE / column CE / L2-normalized embedding shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 16
    temperature: float = 0.15
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(
    image_batch: Tensor,
    text_batch: Tensor,
    apply_noise: bool = False,
) -> tuple[Tensor, Tensor, Tensor]:
    if apply_noise:
        prepared_text, corrupted = noisy_captions(
            text_batch,
            rate=0.3,
            seed=5,
        )
    else:
        prepared_text = text_batch
        corrupted = torch.empty(0, dtype=torch.long)
    return image_batch, prepared_text, corrupted


class AlignPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        vocabulary = int(tokens.max()) + 1
        self.image_encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, config.hidden_dim),
        )
        self.token_embedding = nn.Embedding(vocabulary, config.hidden_dim)
        self.text_projection = nn.Linear(config.hidden_dim, config.hidden_dim)

    def forward(self, image_batch: Tensor, text_batch: Tensor) -> Tensor:
        image_embedding = F.normalize(self.image_encoder(image_batch), dim=1)
        text_hidden = self.token_embedding(text_batch).mean(dim=1)
        text_embedding = F.normalize(self.text_projection(text_hidden), dim=1)
        return image_embedding @ text_embedding.T / self.config.temperature

    def compute_loss(self, similarity: Tensor) -> Tensor:
        target = torch.arange(len(similarity), device=similarity.device)
        return 0.5 * (
            F.cross_entropy(similarity, target) + F.cross_entropy(similarity.T, target)
        )

    def training_step(self, image_batch: Tensor, text_batch: Tensor) -> Tensor:
        return self.compute_loss(self(image_batch, text_batch))

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §2.2, Eq. (1) / §2.2, Eq. (2)
- **이 셀의 책임:** image-to-text normalized softmax / text-to-image symmetric loss
- **Tensor shape 계약:** noisy image/text batch [B,*] → embeddings [B,D] → logits [B,B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** row CE / column CE. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: AlignPortfolioModel,
    image_batch: Tensor,
    text_batch: Tensor,
) -> list[float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(image_batch, text_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §2.1 / §3.2 retrieval evaluation
- **이 셀의 책임:** minimal filtering의 noisy pairs / dual-encoder nearest-neighbor retrieval
- **Tensor shape 계약:** noisy image/text batch [B,*] → embeddings [B,D] → logits [B,B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 오염 index·비율 / clean test Recall@1. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: AlignPortfolioModel,
    image_batch: Tensor,
    clean_text_batch: Tensor,
) -> dict[str, float]:
    with torch.no_grad():
        _, noisy_text_batch, corrupted = prepare_views(
            image_batch,
            clean_text_batch,
            apply_noise=True,
        )
        clean_similarity = model(image_batch, clean_text_batch)
        noisy_similarity = model(image_batch, noisy_text_batch)
        target = torch.arange(
            len(clean_similarity),
            device=clean_similarity.device,
        )
        clean_recall = (clean_similarity.argmax(1) == target).float().mean()
        noisy_recall = (noisy_similarity.argmax(1) == target).float().mean()
    return {
        "clean_image_to_text_recall1": float(clean_recall.cpu()),
        "noisy_image_to_text_recall1": float(noisy_recall.cpu()),
        "caption_noise_rate": len(corrupted) / len(clean_text_batch),
    }


portfolio_train_index = train_idx[:96]
portfolio_test_index = test_idx
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
(
    portfolio_train_images,
    portfolio_noisy_tokens,
    portfolio_corrupted,
) = prepare_views(
    images[portfolio_train_index],
    tokens[portfolio_train_index],
    apply_noise=True,
)
assert len(portfolio_corrupted) == int(len(portfolio_train_index) * 0.3)
portfolio_train_images, portfolio_noisy_tokens = ACCELERATOR.move(
    portfolio_train_images,
    portfolio_noisy_tokens,
)
portfolio_test_images, portfolio_clean_test_tokens = ACCELERATOR.move(
    images[test_idx],
    tokens[test_idx],
)
portfolio_model = ACCELERATOR.move(AlignPortfolioModel(PortfolioConfig()))
portfolio_history = fit_portfolio_model(
    portfolio_model,
    portfolio_train_images,
    portfolio_noisy_tokens,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    portfolio_test_images,
    portfolio_clean_test_tokens,
)
assert np.isfinite(portfolio_history).all()
expected_noise_rate = int(len(test_idx) * 0.3) / len(test_idx)
assert portfolio_metrics["caption_noise_rate"] == expected_noise_rate
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 18억 noisy image-alt-text pairs 대신 240개 중 30%를 결정적으로 오염시킵니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.